# Named Entity Recognition (NER) Model

This notebook demonstrates building a **BiLSTM-LSTM model** for Named Entity Recognition using TensorFlow/Keras.  
It covers **data loading, preprocessing, model building, training, evaluation, and prediction**.

In [15]:
# Cell 1: Imports and setup
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Embedding, Dense, TimeDistributed, Dropout, Bidirectional
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
from itertools import chain
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(1)
tf.random.set_seed(13)

## Load Dataset

Load the NER dataset (`NER dataset.csv`) and perform basic validation.

In [16]:
# Cell 2: Load dataset
try:
    data = pd.read_csv('NER dataset.csv', encoding='unicode_escape')
    print(f"Dataset loaded successfully with {len(data)} rows")
    print(f"Columns: {list(data.columns)}")
except FileNotFoundError:
    print("Error: 'NER dataset.csv' not found. Ensure the file exists in the current directory.")
    exit()
except Exception as e:
    print(f"Error loading dataset: {e}")
    exit()

# Display first few rows
print(data.head())

# Check required columns
required_columns = ['Word', 'Tag', 'Sentence #']
missing_columns = [col for col in required_columns if col not in data.columns]
if missing_columns:
    print(f"Error: Missing required columns: {missing_columns}")
    exit()

Dataset loaded successfully with 1048575 rows
Columns: ['Sentence #', 'Word', 'POS', 'Tag']
    Sentence #           Word  POS Tag
0  Sentence: 1      Thousands  NNS   O
1          NaN             of   IN   O
2          NaN  demonstrators  NNS   O
3          NaN           have  VBP   O
4          NaN        marched  VBN   O


## Create Token & Tag Mappings

In [17]:
# Cell 3: Mapping functions
def get_dict_map(data, token_or_tag):
    """Create token/tag to index mappings"""
    if token_or_tag == 'token':
        vocab = list(set(data['Word'].to_list()))
        vocab = ['<PAD>', '<UNK>'] + vocab
    else:
        vocab = list(set(data['Tag'].to_list()))
        vocab = ['<PAD>'] + vocab
    
    idx2tok = {idx: tok for idx, tok in enumerate(vocab)}
    tok2idx = {tok: idx for idx, tok in enumerate(vocab)}
    return tok2idx, idx2tok

# Create mappings
token2idx, idx2token = get_dict_map(data, 'token')
tag2idx, idx2tag = get_dict_map(data, 'tag')

print(f"Vocabulary size: {len(token2idx)}")
print(f"Number of tags: {len(tag2idx)}")
print(f"Tags: {list(tag2idx.keys())}")


Vocabulary size: 35180
Number of tags: 18
Tags: ['<PAD>', 'B-eve', 'I-org', 'B-per', 'B-geo', 'I-per', 'B-nat', 'I-art', 'I-geo', 'B-org', 'I-tim', 'I-gpe', 'O', 'I-eve', 'B-tim', 'B-gpe', 'B-art', 'I-nat']


## Preprocess Data

In [18]:
# Cell 4: Map words and tags to indices, forward fill missing values
data['Word_idx'] = data['Word'].map(lambda x: token2idx.get(x, token2idx['<UNK>']))
data['Tag_idx'] = data['Tag'].map(tag2idx)

# Forward fill missing values
data_fillna = data.ffill()

# Group by sentence
data_group = data_fillna.groupby(
    ['Sentence #'], as_index=False
)[['Word', 'POS', 'Tag', 'Word_idx', 'Tag_idx']].agg(lambda x: list(x))


## Prepare Train, Validation, and Test Sets

In [19]:
# Cell 5: Padding and splitting
def get_pad_train_test_val(data_group, token2idx, tag2idx):
    """Prepare padded train, val, test sets"""
    n_token = len(token2idx)
    n_tag = len(tag2idx)

    tokens = data_group['Word_idx'].tolist()
    maxlen = max([len(s) for s in tokens])
    pad_tokens = pad_sequences(tokens, maxlen=maxlen, dtype='int32', padding='post', value=token2idx['<PAD>'])

    tags = data_group['Tag_idx'].tolist()
    pad_tags = pad_sequences(tags, maxlen=maxlen, dtype='int32', padding='post', value=tag2idx['<PAD>'])
    pad_tags = [to_categorical(i, num_classes=n_tag) for i in pad_tags]

    # Split into test set first
    tokens_, test_tokens, tags_, test_tags = train_test_split(pad_tokens, pad_tags, train_size=0.8, random_state=42)
    
    # Then split into train and validation
    train_tokens, val_tokens, train_tags, val_tags = train_test_split(tokens_, tags_, train_size=0.8, random_state=42)

    print(f'Train: {len(train_tokens)}, Val: {len(val_tokens)}, Test: {len(test_tokens)}, Max length: {maxlen}')
    return train_tokens, val_tokens, test_tokens, train_tags, val_tags, test_tags, maxlen

# Run the function to create datasets
train_tokens, val_tokens, test_tokens, train_tags, val_tags, test_tags, max_length = get_pad_train_test_val(data_group, token2idx, tag2idx)


Train: 30693, Val: 7674, Test: 9592, Max length: 104


## Build BiLSTM + LSTM Model

In [21]:
# Cell 6: Model definition
input_dim = len(token2idx)
output_dim = 64
n_tags = len(tag2idx)

def get_bilstm_lstm_model(input_length, input_dim, output_dim, n_tags):
    model = Sequential([
        Embedding(input_dim=input_dim, output_dim=output_dim, input_length=input_length),
        Bidirectional(LSTM(units=output_dim, return_sequences=True, dropout=0.2, recurrent_dropout=0.0)),
        LSTM(units=output_dim, return_sequences=True, dropout=0.5, recurrent_dropout=0.0),
        TimeDistributed(Dense(n_tags, activation="softmax"))
    ])
    
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

model = get_bilstm_lstm_model(input_length=max_length, input_dim=input_dim, output_dim=output_dim, n_tags=n_tags)
model.build(input_shape=(None, max_length))  # Force build to show correct summary
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 104, 64)        │     2,251,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 104, 128)       │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 104, 64)        │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 104, 18)        │         1,170 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,368,146 (9.03 MB)

 Trainable params: 2,368,146 (9.03 MB)

 Non-trainable params: 0 (0.00 B)

## Train Model

In [22]:
# Cell 7: Training
history = model.fit(
    np.array(train_tokens), np.array(train_tags),
    validation_data=(np.array(val_tokens), np.array(val_tags)),
    epochs=10,      # Increase for full training
    batch_size=32,
    verbose=1
)


Epoch 1/10
960/960 ━━━━━━━━━━━━━━━━━━━━ 109s 108ms/step - accuracy: 0.9442 - loss: 0.2891 - val_accuracy: 0.9789 - val_loss: 0.0655
Epoch 2/10
960/960 ━━━━━━━━━━━━━━━━━━━━ 107s 112ms/step - accuracy: 0.9844 - loss: 0.0539 - val_accuracy: 0.9917 - val_loss: 0.0302
Epoch 3/10
960/960 ━━━━━━━━━━━━━━━━━━━━ 114s 119ms/step - accuracy: 0.9927 - loss: 0.0270 - val_accuracy: 0.9924 - val_loss: 0.0264
Epoch 4/10
960/960 ━━━━━━━━━━━━━━━━━━━━ 113s 118ms/step - accuracy: 0.9940 - loss: 0.0213 - val_accuracy: 0.9926 - val_loss: 0.0256
Epoch 5/10
960/960 ━━━━━━━━━━━━━━━━━━━━ 116s 121ms/step - accuracy: 0.9947 - loss: 0.0182 - val_accuracy: 0.9928 - val_loss: 0.0257
Epoch 6/10
960/960 ━━━━━━━━━━━━━━━━━━━━ 116s 121ms/step - accuracy: 0.9953 - loss: 0.0159 - val_accuracy: 0.9928 - val_loss: 0.0264
Epoch 7/10
960/960 ━━━━━━━━━━━━━━━━━━━━ 113s 118ms/step - accuracy: 0.9957 - loss: 0.0142 - val_accuracy: 0.9927 - val_loss: 0.0281
Epoch 8/10
960/960 ━━━━━━━━━━━━━━━━━━━━ 103s 107ms/step - accuracy: 0.9962 -

## Evaluate Model

In [26]:
# Cell 8: Evaluation
test_loss, test_accuracy = model.evaluate(np.array(test_tokens), np.array(test_tags), verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


Test Loss: 0.0307
Test Accuracy: 0.9926


## Predict Named Entities

In [25]:
# Cell 9: Prediction function
def predict_entities(text, model, token2idx, idx2tag, max_length):
    words = text.split()
    word_indices = [token2idx.get(word, token2idx['<UNK>']) for word in words]
    if len(word_indices) < max_length:
        word_indices += [token2idx['<PAD>']] * (max_length - len(word_indices))
    else:
        word_indices = word_indices[:max_length]
    X = np.array([word_indices])
    predictions = model.predict(X)
    predicted_tags = np.argmax(predictions[0], axis=1)
    entities = [(word, idx2tag[idx]) for word, idx in zip(words, predicted_tags) if idx2tag[idx] != '<PAD>']
    return entities

# Example usage
sample_text = "John Smith works at Microsoft in Seattle"
entities = predict_entities(sample_text, model, token2idx, idx2tag, max_length)
print(f"Text: {sample_text}")
print(f"Predicted entities: {entities}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 819ms/step
Text: John Smith works at Microsoft in Seattle
Predicted entities: [('John', 'B-per'), ('Smith', 'I-per'), ('works', 'O'), ('at', 'O'), ('Microsoft', 'B-org'), ('in', 'O'), ('Seattle', 'B-geo')]
